# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [6]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [7]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [8]:
#fix cabin granularity
spaceship.columns=spaceship.columns.str.lower()
spaceship['cabin']=spaceship['cabin'].str[0]

In [9]:
#drop passenger id and name cols
spaceship=spaceship.drop(['passengerid', 'name'], axis=1)

In [10]:
#for non numerical columns, do dummies
#select features columns, discard transported column
cat_cols=spaceship.select_dtypes(exclude='number').columns
cat_cols=cat_cols.drop(['transported'])
cat_cols

Index(['homeplanet', 'cryosleep', 'cabin', 'destination', 'vip'], dtype='object')

In [11]:
#let´s do dummies
spaceship_dummies=pd.get_dummies(spaceship, columns=cat_cols)

In [12]:
spaceship

,homeplanet,cryosleep,cabin,destination,age,vip,roomservice,foodcourt,shoppingmall,spa,vrdeck,transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...
8688,Europa,False,A,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,False
8689,Earth,True,G,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,False
8690,Earth,False,G,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,True
8691,Europa,False,E,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,False


**Perform Train Test Split**

In [34]:
target=spaceship['transported']
features=spaceship_dummies.drop(['transported'], axis=1)

In [44]:
features.columns

Index(['age', 'roomservice', 'foodcourt', 'shoppingmall', 'spa', 'vrdeck',
       'homeplanet_Earth', 'homeplanet_Europa', 'homeplanet_Mars',
       'cryosleep_False', 'cryosleep_True', 'cabin_A', 'cabin_B', 'cabin_C',
       'cabin_D', 'cabin_E', 'cabin_F', 'cabin_G', 'cabin_T',
       'destination_55 Cancri e', 'destination_PSO J318.5-22',
       'destination_TRAPPIST-1e', 'vip_False', 'vip_True'],
      dtype='object')

In [15]:
X_train, X_test, y_train, y_test= train_test_split(features, target, test_size=0.20, random_state=0)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [16]:
# import tree model, bagging classifier and score metrics
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, classification_report

In [60]:
# base estimator: 

base_tree = DecisionTreeClassifier(
    max_depth=20,     
    random_state=42)

# 2. bagging wrapper

bagging_clf= BaggingClassifier( estimator=base_tree, n_estimators=100, max_samples=1000, max_features=1.0, bootstrap=True, n_jobs=-1, random_state=42)

In [18]:
#train

In [61]:
bagging_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=20,
                                                   random_state=42),
                  max_samples=1000, n_estimators=100, n_jobs=-1,
                  random_state=42)

In [63]:
y_pred_bag = bagging_clf.predict(X_test)
print("Bagging accuracy:", accuracy_score(y_test, y_pred_bag))

Bagging accuracy: 0.7958596894767107


- Random Forests

In [64]:
#your code here

from sklearn.ensemble import RandomForestClassifier
rf_cf=RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)


In [65]:
#fit model
rf_cf.fit(X_train, y_train)

RandomForestClassifier(max_depth=20, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [66]:
#apply model
y_pred = rf_cf.predict(X_test)

In [67]:
accuracy_score(y_test, y_pred)

0.7837837837837838

- Adaptive Boosting

In [41]:
from sklearn.ensemble import AdaBoostClassifier

In [56]:
#initialize AdaBoost Model
ada_clas= AdaBoostClassifier(estimator=base_tree, n_estimators=100)

In [ ]:


# treat the null values

from sklearn.impute import SimpleImputer

#Identify numeric cols in our train split
num_cols=X_train.select_dtypes(include='float64').columns

#impute median
imputer = SimpleImputer(strategy='median')

#fit the imputter to our training data
X_train[num_cols]=imputer.fit_transform(X_train[num_cols])

#apply same to test split
X_test[num_cols]=imputer.transform(X_test[num_cols])

In [57]:
#Training the Model
ada_clas.fit(X_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=42),
                   n_estimators=100)

In [58]:
#apply our model to predict
adaboost_pred= ada_clas.predict(X_test)

In [68]:
#let´s see the score

accuracy_score(y_test, adaboost_pred)

0.7527314548591144

- Gradient Boosting

In [69]:
from sklearn.ensemble import GradientBoostingClassifier

In [70]:
#your code here
gb_class= GradientBoostingClassifier(max_depth=20, n_estimators=100)

In [71]:
#train model

gb_class.fit(X_train, y_train)

GradientBoostingClassifier(max_depth=20)

In [ ]:
#evaluate model
gb_pred=gb_class.predict(X_test)

In [73]:
#Score

accuracy_score(y_test, gb_pred)

0.7665324899367453

Which model is the best and why?

In [ ]:
#best model is Tree Decision classifier + Bagging Ensemble Method 